# RNN-Modell

## 1. Datenvorbereitung

Die Datenvorbereitung entspricht exakt derjenigen aus `02_word_embeddings_model.ipynb`. Es werden dieselben Datensplits, dieselbe Tokenisierung und dieselbe Vokabular-Erstellung aus `src/preprocessing.py` verwendet. Dadurch sind die Ergebnisse des RNN-Modells direkt mit dem Word-Embedding-Modell vergleichbar.

In [1]:
import torch
import torch.nn as nn

from datasets import load_dataset

### 1.1 Datensatz laden

Es werden ausschliesslich der Trainings- und der Validierungsdatensatz geladen.

Der Testdatensatz wird erst nach Abschluss der Modellentwicklung für die abschliessende Bewertung verwendet.

In [2]:
train = load_dataset("amarfurt/art-nlp-hs26", split="train")
valid = load_dataset("amarfurt/art-nlp-hs26", split="validation")

print("Trainingsbeispiele:", len(train))
print("Validierungsbeispiele:", len(valid))

Trainingsbeispiele: 16801
Validierungsbeispiele: 1000


### 1.2 Preprocessing-Funktionen importieren

Das Projektverzeichnis wird zum Python-Suchpfad hinzugefügt, damit die bestehenden Funktionen aus `src/preprocessing.py` verwendet werden können.

In [3]:
import sys
from pathlib import Path

# Projektverzeichnis zum Python-Suchpfad hinzufügen
project_root = Path.cwd().resolve()

if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.preprocessing import build_vocab, encode_example, collate_batch

### 1.3 Vokabular erstellen

Das Vokabular wird wie im Word-Embedding-Modell ausschliesslich aus den Trainingsdaten mit `min_freq=1` erstellt.

In [4]:
vocab = build_vocab(train, min_freq=1)

print("Vokabulargrösse:", len(vocab))
print("ID von '<PAD>':", vocab["<PAD>"])
print("ID von '<UNK>':", vocab["<UNK>"])

Vokabulargrösse: 15818
ID von '<PAD>': 0
ID von '<UNK>': 1


### 1.4 Kodierung und DataLoader

Die Trainings- und Validierungsbeispiele werden mit `encode_example()` in Token-IDs umgewandelt.

Anschliessend werden DataLoader mit einer Batch Size von 256 erstellt. Die Trainingsdaten werden gemischt, die Validierungsdaten bleiben in ihrer ursprünglichen Reihenfolge. `collate_batch()` ergänzt unterschiedlich lange Sequenzen innerhalb eines Batches mit Padding.

In [5]:
from torch.utils.data import DataLoader

# Zufallszahlen für reproduzierbares Mischen
torch.manual_seed(42)

# Trainings- und Validierungsdaten kodieren
train_encoded = [encode_example(example, vocab) for example in train]
valid_encoded = [encode_example(example, vocab) for example in valid]

batch_size = 256

train_loader = DataLoader(
    train_encoded,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_batch
)

valid_loader = DataLoader(
    valid_encoded,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_batch
)

print("Trainings-Batches:", len(train_loader))
print("Validierungs-Batches:", len(valid_loader))

Trainings-Batches: 66
Validierungs-Batches: 4


### 1.5 Test des RNN-Modells mit einem echten Batch

Zur Überprüfung wird ein Batch aus den Trainingsdaten durch das `RNNModel` geschickt. Es findet noch kein Training statt.

Das Modell muss für jedes Beispiel zwei Scores zurückgeben, die Ausgabe hat also die Form `[batch_size, 2]`.

In [6]:
from src.models import RNNModel

# Überprüfen, ob das Padding-Token den Index 0 hat
assert vocab["<PAD>"] == 0

# Reproduzierbarkeit
torch.manual_seed(42)

# GPU verwenden, falls verfügbar
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# RNN-Modell erstellen und auf GPU verschieben
rnn_model = RNNModel(
    vocab_size=len(vocab),
    embedding_dim=50,
    hidden_dim=64
).to(device)

# Einen echten Batch aus dem Trainingsdatensatz laden
batch = next(iter(train_loader))

# Batch auf dasselbe Gerät wie das Modell verschieben
batch = {
    key: value.to(device)
    for key, value in batch.items()
}

# Modell in den Evaluationsmodus versetzen
rnn_model.eval()

# Forward Pass ohne Gradientenberechnung
with torch.no_grad():
    logits = rnn_model(batch)

# Ausgabe überprüfen
print("Device:", device)
print("Batch Size:", batch["label"].size(0))
print("Logits Shape:", logits.shape)

assert logits.shape == (batch["label"].size(0), 2)

print("Forward-Pass-Test erfolgreich!")

Device: cuda
Batch Size: 256
Logits Shape: torch.Size([256, 2])
Forward-Pass-Test erfolgreich!


### 2.1 RNN Training

Das RNN-Modell wird mit derselben Experimentfunktion trainiert wie das Word-Embedding-Modell. Über den Parameter `model_type="rnn"` wird das `RNNModel` verwendet.

Für den ersten Trainingslauf werden die besten Hyperparameter aus der Optimierung des Word-Embedding-Modells übernommen (Batch Size 256, Learning Rate 0.001, Weight Decay 0.001, kein Dropout). Early Stopping, Checkpointing, W&B-Logging und Caching funktionieren wie bisher. RNN-Experimente erhalten eigene Checkpoints, Ergebnisdateien und W&B-Runs.

Das Training wird nur ausgeführt, wenn `RUN_RNN_TRAINING = True` gesetzt ist.

In [ ]:
import importlib
import src.models
import src.training

# Aktualisierte Python-Dateien neu laden
importlib.reload(src.models)
importlib.reload(src.training)

from src.training import run_experiment

# Schalter für das RNN-Training
# False: Die Trainingszelle wird übersprungen (kein Training, kein W&B-Run).
# True: Das RNN-Modell wird trainiert (bereits gespeicherte Ergebnisse werden aus dem Cache geladen).
RUN_RNN_TRAINING = False

In [10]:
if RUN_RNN_TRAINING:
    result_rnn_baseline = run_experiment(
        train_encoded=train_encoded,
        valid_encoded=valid_encoded,
        vocab=vocab,
        project_root=project_root,
        device=device,
        batch_size=256,
        learning_rate=0.001,
        epochs=60,
        patience=5,
        seed=42,
        dropout=0.0,
        weight_decay=0.001,
        model_type="rnn"
    )
else:
    print("RNN-Training übersprungen (RUN_RNN_TRAINING = False).")

wandb: Loading settings from C:\Users\Marco\.config\wandb\settings
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\Marco\_netrc.
wandb: Currently logged in as: marco-blum (marco-blum-hochschule-luzern) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


BS 256 | DO 0.0 | Epoch 1/60 | Train Acc: 54.22% | Valid Acc: 55.70%
BS 256 | DO 0.0 | Epoch 2/60 | Train Acc: 60.80% | Valid Acc: 62.50%
BS 256 | DO 0.0 | Epoch 3/60 | Train Acc: 65.63% | Valid Acc: 63.50%
BS 256 | DO 0.0 | Epoch 4/60 | Train Acc: 69.35% | Valid Acc: 63.90%
BS 256 | DO 0.0 | Epoch 5/60 | Train Acc: 72.76% | Valid Acc: 64.20%
BS 256 | DO 0.0 | Epoch 6/60 | Train Acc: 75.70% | Valid Acc: 62.00%
BS 256 | DO 0.0 | Epoch 7/60 | Train Acc: 78.93% | Valid Acc: 64.40%
BS 256 | DO 0.0 | Epoch 8/60 | Train Acc: 81.15% | Valid Acc: 63.40%
BS 256 | DO 0.0 | Epoch 9/60 | Train Acc: 84.03% | Valid Acc: 64.20%
BS 256 | DO 0.0 | Epoch 10/60 | Train Acc: 86.52% | Valid Acc: 63.90%
BS 256 | DO 0.0 | Epoch 11/60 | Train Acc: 89.05% | Valid Acc: 64.30%
BS 256 | DO 0.0 | Epoch 12/60 | Train Acc: 91.11% | Valid Acc: 65.10%
BS 256 | DO 0.0 | Epoch 13/60 | Train Acc: 93.19% | Valid Acc: 64.70%
BS 256 | DO 0.0 | Epoch 14/60 | Train Acc: 94.96% | Valid Acc: 63.90%
BS 256 | DO 0.0 | Epoch 15/60

epoch,▁▁▂▂▃▃▄▄▅▅▅▆▆▇▇██
train_accuracy,▁▂▃▃▄▄▅▅▆▆▇▇▇▇███
train_loss,██▇▇▆▆▅▅▄▄▃▃▂▂▂▁▁
valid_accuracy,▁▆▇▇▇▆▇▇▇▇▇██▇▇▇▇
valid_loss,▁▁▁▁▁▁▁▁▂▂▂▃▃▄▅▇█
best_epoch,12
best_valid_accuracy,0.651
epoch,17
train_accuracy,0.98423
train_loss,0.04734
training_time_seconds,63.19623


### 2.2 RNN Dropout Experiments

Zur Untersuchung der Regularisierung wird das RNN-Modell zusätzlich mit den Dropout-Raten 0.2 und 0.5 trainiert. Dropout wird im Klassifikator nach der ReLU-Aktivierung angewendet.

Alle übrigen Hyperparameter bleiben gegenüber dem ersten RNN-Trainingslauf unverändert (Batch Size 256, Learning Rate 0.001, Weight Decay 0.001, maximal 60 Epochen, Patience 5, Seed 42). Dadurch lassen sich die Ergebnisse direkt mit dem Lauf ohne Dropout vergleichen.

W&B-Logging, Checkpointing und Caching funktionieren wie bisher. Die Experimente werden nur ausgeführt, wenn `RUN_RNN_DROPOUT = True` gesetzt ist.

In [ ]:
# Schalter für die RNN-Dropout-Experimente
# False: Die Experimentzellen werden übersprungen (kein Training, keine W&B-Runs).
# True: Die Experimente werden ausgeführt (bereits gespeicherte Ergebnisse werden aus dem Cache geladen).
RUN_RNN_DROPOUT = False

In [12]:
if RUN_RNN_DROPOUT:
    result_rnn_dropout_02 = run_experiment(
        train_encoded=train_encoded,
        valid_encoded=valid_encoded,
        vocab=vocab,
        project_root=project_root,
        device=device,
        batch_size=256,
        learning_rate=0.001,
        epochs=60,
        patience=5,
        seed=42,
        dropout=0.2,
        weight_decay=0.001,
        model_type="rnn"
    )
else:
    print("result_rnn_dropout_02 übersprungen (RUN_RNN_DROPOUT = False).")

BS 256 | DO 0.2 | Epoch 1/60 | Train Acc: 52.01% | Valid Acc: 56.30%
BS 256 | DO 0.2 | Epoch 2/60 | Train Acc: 57.94% | Valid Acc: 60.10%
BS 256 | DO 0.2 | Epoch 3/60 | Train Acc: 63.25% | Valid Acc: 61.50%
BS 256 | DO 0.2 | Epoch 4/60 | Train Acc: 67.41% | Valid Acc: 61.10%
BS 256 | DO 0.2 | Epoch 5/60 | Train Acc: 71.02% | Valid Acc: 63.30%
BS 256 | DO 0.2 | Epoch 6/60 | Train Acc: 73.58% | Valid Acc: 62.80%
BS 256 | DO 0.2 | Epoch 7/60 | Train Acc: 76.64% | Valid Acc: 64.40%
BS 256 | DO 0.2 | Epoch 8/60 | Train Acc: 78.86% | Valid Acc: 64.30%
BS 256 | DO 0.2 | Epoch 9/60 | Train Acc: 81.76% | Valid Acc: 64.50%
BS 256 | DO 0.2 | Epoch 10/60 | Train Acc: 84.10% | Valid Acc: 64.00%
BS 256 | DO 0.2 | Epoch 11/60 | Train Acc: 86.37% | Valid Acc: 65.30%
BS 256 | DO 0.2 | Epoch 12/60 | Train Acc: 88.85% | Valid Acc: 65.20%
BS 256 | DO 0.2 | Epoch 13/60 | Train Acc: 90.55% | Valid Acc: 64.90%
BS 256 | DO 0.2 | Epoch 14/60 | Train Acc: 92.47% | Valid Acc: 65.20%
BS 256 | DO 0.2 | Epoch 15/60

epoch,▁▁▂▂▂▃▃▃▄▄▄▅▅▅▆▆▆▇▇▇██
train_accuracy,▁▂▃▃▄▄▅▅▅▆▆▆▇▇▇▇██████
train_loss,██▇▇▇▆▆▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁
valid_accuracy,▁▄▅▅▆▆▇▇▇▇█▇▇▇▇██▇▇▇▇▇
valid_loss,▁▁▁▁▁▁▁▁▁▂▂▂▂▃▄▄▅▅▆▇▇█
best_epoch,17
best_valid_accuracy,0.659
epoch,22
train_accuracy,0.98941
train_loss,0.03214
training_time_seconds,74.73731


In [13]:
if RUN_RNN_DROPOUT:
    result_rnn_dropout_05 = run_experiment(
        train_encoded=train_encoded,
        valid_encoded=valid_encoded,
        vocab=vocab,
        project_root=project_root,
        device=device,
        batch_size=256,
        learning_rate=0.001,
        epochs=60,
        patience=5,
        seed=42,
        dropout=0.5,
        weight_decay=0.001,
        model_type="rnn"
    )
else:
    print("result_rnn_dropout_05 übersprungen (RUN_RNN_DROPOUT = False).")

BS 256 | DO 0.5 | Epoch 1/60 | Train Acc: 50.47% | Valid Acc: 56.50%
BS 256 | DO 0.5 | Epoch 2/60 | Train Acc: 52.82% | Valid Acc: 55.70%
BS 256 | DO 0.5 | Epoch 3/60 | Train Acc: 59.05% | Valid Acc: 60.80%
BS 256 | DO 0.5 | Epoch 4/60 | Train Acc: 63.37% | Valid Acc: 61.10%
BS 256 | DO 0.5 | Epoch 5/60 | Train Acc: 66.98% | Valid Acc: 62.80%
BS 256 | DO 0.5 | Epoch 6/60 | Train Acc: 70.54% | Valid Acc: 63.30%
BS 256 | DO 0.5 | Epoch 7/60 | Train Acc: 73.07% | Valid Acc: 63.20%
BS 256 | DO 0.5 | Epoch 8/60 | Train Acc: 75.69% | Valid Acc: 64.50%
BS 256 | DO 0.5 | Epoch 9/60 | Train Acc: 78.07% | Valid Acc: 64.60%
BS 256 | DO 0.5 | Epoch 10/60 | Train Acc: 80.27% | Valid Acc: 64.30%
BS 256 | DO 0.5 | Epoch 11/60 | Train Acc: 82.71% | Valid Acc: 65.40%
BS 256 | DO 0.5 | Epoch 12/60 | Train Acc: 85.05% | Valid Acc: 64.40%
BS 256 | DO 0.5 | Epoch 13/60 | Train Acc: 87.15% | Valid Acc: 65.70%
BS 256 | DO 0.5 | Epoch 14/60 | Train Acc: 88.91% | Valid Acc: 65.70%
BS 256 | DO 0.5 | Epoch 15/60

epoch,▁▁▂▂▂▃▃▃▄▄▄▅▅▅▆▆▆▇▇▇██
train_accuracy,▁▁▂▃▃▄▄▅▅▅▆▆▇▇▇▇▇█████
train_loss,███▇▇▆▆▆▅▅▄▄▃▃▃▂▂▂▁▁▁▁
valid_accuracy,▂▁▄▅▆▆▆▇▇▇█▇██▇▇████▇▇
valid_loss,▁▁▁▁▁▁▁▁▁▁▂▂▂▃▄▄▅▅▆▇██
best_epoch,17
best_valid_accuracy,0.661
epoch,22
train_accuracy,0.97072
train_loss,0.07831
training_time_seconds,76.32283


### 2.3 RNN Seed Experiments

Um die Stabilität der Ergebnisse zu untersuchen, wird die beste bisherige RNN-Konfiguration (Batch Size 256, Learning Rate 0.001, Dropout 0.5, Weight Decay 0.001, maximal 60 Epochen, Patience 5) zusätzlich mit den Random Seeds 123 und 456 trainiert.

Zusammen mit dem Lauf mit Seed 42 aus Abschnitt 3.5 lässt sich so abschätzen, wie stark die Validation Accuracy von der zufälligen Initialisierung und der Reihenfolge der Trainingsbeispiele abhängt.

W&B-Logging, Checkpointing und Caching funktionieren wie bisher. Die Experimente werden nur ausgeführt, wenn `RUN_RNN_SEEDS = True` gesetzt ist.

In [ ]:
# Schalter für die RNN-Seed-Experimente
# False: Die Experimentzellen werden übersprungen (kein Training, keine W&B-Runs).
# True: Die Experimente werden ausgeführt (bereits gespeicherte Ergebnisse werden aus dem Cache geladen).
RUN_RNN_SEEDS = False

In [15]:
if RUN_RNN_SEEDS:
    result_rnn_seed_123 = run_experiment(
        train_encoded=train_encoded,
        valid_encoded=valid_encoded,
        vocab=vocab,
        project_root=project_root,
        device=device,
        batch_size=256,
        learning_rate=0.001,
        epochs=60,
        patience=5,
        seed=123,
        dropout=0.5,
        weight_decay=0.001,
        model_type="rnn"
    )
else:
    print("result_rnn_seed_123 übersprungen (RUN_RNN_SEEDS = False).")

BS 256 | DO 0.5 | Epoch 1/60 | Train Acc: 50.53% | Valid Acc: 51.80%
BS 256 | DO 0.5 | Epoch 2/60 | Train Acc: 52.98% | Valid Acc: 54.60%
BS 256 | DO 0.5 | Epoch 3/60 | Train Acc: 59.28% | Valid Acc: 57.40%
BS 256 | DO 0.5 | Epoch 4/60 | Train Acc: 63.74% | Valid Acc: 61.80%
BS 256 | DO 0.5 | Epoch 5/60 | Train Acc: 67.95% | Valid Acc: 62.90%
BS 256 | DO 0.5 | Epoch 6/60 | Train Acc: 70.75% | Valid Acc: 63.70%
BS 256 | DO 0.5 | Epoch 7/60 | Train Acc: 73.76% | Valid Acc: 64.30%
BS 256 | DO 0.5 | Epoch 8/60 | Train Acc: 76.16% | Valid Acc: 63.00%
BS 256 | DO 0.5 | Epoch 9/60 | Train Acc: 78.64% | Valid Acc: 62.40%
BS 256 | DO 0.5 | Epoch 10/60 | Train Acc: 80.80% | Valid Acc: 63.70%
BS 256 | DO 0.5 | Epoch 11/60 | Train Acc: 83.19% | Valid Acc: 63.20%
BS 256 | DO 0.5 | Epoch 12/60 | Train Acc: 85.30% | Valid Acc: 62.90%
Early Stopping nach Epoche 12

Beste Validation Accuracy: 64.30%
Beste Epoche: 7
Trainingszeit: 41.8 Sekunden


epoch,▁▂▂▃▄▄▅▅▆▇▇█
train_accuracy,▁▁▃▄▅▅▆▆▇▇██
train_loss,██▇▇▆▅▅▄▃▃▂▁
valid_accuracy,▁▃▄▇▇██▇▇█▇▇
valid_loss,▂▂▂▁▁▁▂▂▃▅▆█
best_epoch,7
best_valid_accuracy,0.643
epoch,12
train_accuracy,0.85298
train_loss,0.32961
training_time_seconds,41.84974


In [16]:
if RUN_RNN_SEEDS:
    result_rnn_seed_456 = run_experiment(
        train_encoded=train_encoded,
        valid_encoded=valid_encoded,
        vocab=vocab,
        project_root=project_root,
        device=device,
        batch_size=256,
        learning_rate=0.001,
        epochs=60,
        patience=5,
        seed=456,
        dropout=0.5,
        weight_decay=0.001,
        model_type="rnn"
    )
else:
    print("result_rnn_seed_456 übersprungen (RUN_RNN_SEEDS = False).")

BS 256 | DO 0.5 | Epoch 1/60 | Train Acc: 50.68% | Valid Acc: 54.10%
BS 256 | DO 0.5 | Epoch 2/60 | Train Acc: 54.94% | Valid Acc: 57.30%
BS 256 | DO 0.5 | Epoch 3/60 | Train Acc: 61.03% | Valid Acc: 60.70%
BS 256 | DO 0.5 | Epoch 4/60 | Train Acc: 65.08% | Valid Acc: 62.10%
BS 256 | DO 0.5 | Epoch 5/60 | Train Acc: 67.94% | Valid Acc: 63.10%
BS 256 | DO 0.5 | Epoch 6/60 | Train Acc: 71.13% | Valid Acc: 62.80%
BS 256 | DO 0.5 | Epoch 7/60 | Train Acc: 74.03% | Valid Acc: 63.10%
BS 256 | DO 0.5 | Epoch 8/60 | Train Acc: 76.32% | Valid Acc: 64.50%
BS 256 | DO 0.5 | Epoch 9/60 | Train Acc: 78.72% | Valid Acc: 64.00%
BS 256 | DO 0.5 | Epoch 10/60 | Train Acc: 81.23% | Valid Acc: 64.90%
BS 256 | DO 0.5 | Epoch 11/60 | Train Acc: 83.59% | Valid Acc: 64.30%
BS 256 | DO 0.5 | Epoch 12/60 | Train Acc: 85.47% | Valid Acc: 64.60%
BS 256 | DO 0.5 | Epoch 13/60 | Train Acc: 87.61% | Valid Acc: 64.40%
BS 256 | DO 0.5 | Epoch 14/60 | Train Acc: 89.55% | Valid Acc: 63.50%
BS 256 | DO 0.5 | Epoch 15/60

epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
train_accuracy,▁▂▃▄▄▅▅▅▆▆▇▇▇██
train_loss,██▇▇▆▆▅▅▄▄▃▃▂▁▁
valid_accuracy,▁▃▅▆▇▇▇█▇████▇▇
valid_loss,▂▁▁▁▁▁▁▁▂▃▃▄▅▇█
best_epoch,10
best_valid_accuracy,0.649
epoch,15
train_accuracy,0.90959
train_loss,0.21832
training_time_seconds,50.85021


### 2.4 Results and Interpretation

Die Ergebnisse der bisherigen RNN-Experimente werden direkt aus den gespeicherten JSON-Dateien in `outputs/results/` geladen. Dadurch ist die Auswertung reproduzierbar und funktioniert auch nach einem Kernel-Neustart, ohne dass ein Modell erneut trainiert werden muss.

In [19]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

# Projektverzeichnis bestimmen (funktioniert auch nach einem Kernel-Neustart)
results_root = Path.cwd().resolve()

if results_root.name == "notebooks":
    results_root = results_root.parent

results_dir = results_root / "outputs" / "results"

# Bisherige RNN-Experimente: (Bezeichnung, Dropout, Seed)
rnn_experiments = [
    ("Baseline", 0.0, 42),
    ("Dropout 0.2", 0.2, 42),
    ("Dropout 0.5", 0.5, 42),
    ("Seed 123", 0.5, 123),
    ("Seed 456", 0.5, 456),
]

rows = []

for name, dropout, seed in rnn_experiments:
    # Gleiche Experiment-ID wie in run_experiment()
    experiment_id = (
        f"rnn_bs256_lr{0.001:g}_do{dropout:g}_"
        f"wd{0.001:g}_ep60_p5_s{seed}"
    )

    result = json.loads(
        (results_dir / f"{experiment_id}.json").read_text(encoding="utf-8")
    )["result"]

    rows.append({
        "Experiment": name,
        "Dropout": dropout,
        "Seed": seed,
        "Best Validation Accuracy": result["best_valid_accuracy"] * 100,
        "Best Epoch": result["best_epoch"],
    })

rnn_results = pd.DataFrame(rows)

rnn_results.style.format({
    "Dropout": "{:.1f}",
    "Best Validation Accuracy": "{:.2f} %",
}).hide(axis="index")

Experiment,Dropout,Seed,Best Validation Accuracy,Best Epoch
Baseline,0.0,42,65.10 %,12
Dropout 0.2,0.2,42,65.90 %,17
Dropout 0.5,0.5,42,66.10 %,17
Seed 123,0.5,123,64.30 %,7
Seed 456,0.5,456,64.90 %,10


In [20]:
# Beste RNN-Konfiguration (Dropout 0.5) über die drei Seeds
best_config = rnn_results[rnn_results["Dropout"] == 0.5]

mean_acc = best_config["Best Validation Accuracy"].mean()
std_acc = np.std(best_config["Best Validation Accuracy"], ddof=0)  # Population SD

print("Dropout 0.5, Seeds:", ", ".join(str(s) for s in best_config["Seed"]))
print(f"Durchschnittliche Validation Accuracy: {mean_acc:.2f} %")
print(f"Standardabweichung (Population SD): {std_acc:.2f} Prozentpunkte")

Dropout 0.5, Seeds: 42, 123, 456
Durchschnittliche Validation Accuracy: 65.10 %
Standardabweichung (Population SD): 0.75 Prozentpunkte


**Interpretation**

- **Dropout:** Mit Dropout 0.5 steigt die Validation Accuracy bei Seed 42 von 65.10 % (Baseline) auf 66.10 %, also um 1.0 Prozentpunkt. Dropout 0.2 liegt mit 65.90 % dazwischen.
- **Overfitting:** Das RNN zeigt weiterhin deutliches Overfitting. Mit Dropout 0.5 erreicht das Modell in der besten Epoche eine Training Accuracy von rund 94 %, die Validation Accuracy liegt jedoch nur bei 66 %. Die beste Epoche wird zudem früh erreicht (Epoche 7 bis 17), danach sinkt die Validation Accuracy wieder.
- **Seeds:** Die Ergebnisse unterscheiden sich zwischen den Seeds (64.30 % bis 66.10 %). Über die drei Seeds beträgt die durchschnittliche Validation Accuracy 65.10 % mit einer Standardabweichung von ca. 0.75 Prozentpunkten. Der Durchschnitt entspricht damit genau dem Baseline-Wert. Die Verbesserung durch Dropout liegt somit in der Grössenordnung der Schwankung zwischen den Seeds und sollte vorsichtig interpretiert werden.
- **Vergleich mit dem Word-Embedding-Modell:** Das bisher beste Word-Embedding-Modell erreicht 67.80 % Validation Accuracy und liegt damit über dem besten RNN-Ergebnis von 66.10 %. Das komplexere RNN bringt in der aktuellen Konfiguration also keinen Vorteil.